# Transfer learning: phân loại 37 giống chó mèo Oxford-IIIT Pet

Oxford-IIIT Pet: 7 349 ảnh chó mèo thật, cỡ và bố cục đủ kiểu, **37 giống**, khoảng 100 ảnh train mỗi giống.
Với chừng ấy nhãn, train ResNet từ đầu gần như vô vọng; dùng lại một backbone đã pretrain trên ImageNet là
cách làm chuẩn. Bạn dựng đủ các mảnh của một pipeline transfer learning: đọc dữ liệu, transform train/eval
khớp với weights pretrain, thay head, đóng băng backbone (kể cả BatchNorm), learning rate riêng cho backbone
và head, rồi so ba chiến lược có đối chứng: **linear probe**, **fine-tune**, **train từ đầu**.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Điền vào chỗ `# TODO`. Test dùng ResNet-18 khởi tạo ngẫu nhiên
  (`weights=None`), không tải gì.
- Cell `explore` tải bộ dữ liệu (754 MB, lần đầu) và weights ImageNet của ResNet-18 (45 MB, vào
  `~/.cache/torch`), rồi chạy thí nghiệm. Trên CPU cả notebook mất khoảng 15 phút; có GPU thì đặt
  `DEVICE = "cuda"` (kernel `ai-gpu`), chỉ còn khoảng 2 phút.

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, tải Oxford-IIIT Pet nếu chưa có (754 MB, chỉ lần đầu)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

PETS = ensure("oxford-pets")
print(PETS, sorted(p.name for p in (PETS / "data").iterdir()))

In [ ]:
import io
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2

IMAGENET_MEAN = (0.485, 0.456, 0.406)   # thống kê mà weights ImageNet của torchvision đã thấy lúc train
IMAGENET_STD = (0.229, 0.224, 0.225)

## 1. Dữ liệu

Bản mirror Hugging Face (`timm/oxford-iiit-pet`) có hai tệp `data/train-00000-of-00001.parquet` (3 680 ảnh) và
`data/test-00000-of-00001.parquet` (3 669 ảnh), cột `image` là dict `{"bytes": <JPEG/PNG>}`, `label` (0–36),
`image_id` (vd. `Maine_Coon_204`: tên giống + số thứ tự), `label_cat_dog` (0 mèo, 1 chó).

- `load_pets(root, split) -> (images, labels, image_ids)`: `images` danh sách `bytes` (chưa giải mã: giải
  mã cả 7 349 ảnh vào RAM tốn vài GB), `labels` mảng `int64`, `image_ids` danh sách `str`.
- `class_names(labels, image_ids) -> list[str]`: tên giống của lớp 0..C-1 lấy từ `image_id` (bỏ `_<số>` ở
  cuối, giữ nguyên chữ hoa/thường và dấu gạch dưới bên trong, vd. `"Maine_Coon"`, `"american_bulldog"`).
- `PetsDataset(images, labels, transform)`: `Dataset` của PyTorch; `__getitem__(i)` giải mã ảnh `i` thành PIL
  **RGB** (có ảnh gốc là RGBA hoặc thang xám), áp `transform`, trả `(tensor, int(label))`.

In [ ]:
def load_pets(root: Path, split: str) -> tuple[list[bytes], np.ndarray, list[str]]:
    """Đọc data/{split}-00000-of-00001.parquet → (images bytes, labels int64, image_ids)."""
    # TODO
    raise NotImplementedError


def class_names(labels: np.ndarray, image_ids: list[str]) -> list[str]:
    """Tên lớp 0..C-1 từ image_id (bỏ hậu tố _<số>)."""
    # TODO
    raise NotImplementedError


class PetsDataset(Dataset):
    def __init__(self, images: list[bytes], labels: np.ndarray, transform):
        # TODO
        raise NotImplementedError

    def __len__(self) -> int:
        # TODO
        raise NotImplementedError

    def __getitem__(self, i: int):
        # TODO: Image.open(io.BytesIO(...)).convert("RGB")
        raise NotImplementedError

In [ ]:
# explore: đọc dữ liệu thật, số ảnh mỗi lớp, cỡ ảnh, xem vài ảnh
import matplotlib.pyplot as plt

tr_images, tr_labels, tr_ids = load_pets(PETS, "train")
te_images, te_labels, te_ids = load_pets(PETS, "test")
NAMES = class_names(tr_labels, tr_ids)
counts = np.bincount(tr_labels)
print(f"train {len(tr_images)}, test {len(te_images)}, {len(NAMES)} lớp; ảnh/lớp train: min {counts.min()}, max {counts.max()}")
print("vài lớp:", NAMES[:5])
sizes = np.array([Image.open(io.BytesIO(b)).size for b in tr_images[:500]])
print("cỡ ảnh (500 ảnh đầu): rộng", np.percentile(sizes[:, 0], [5, 50, 95]), "cao", np.percentile(sizes[:, 1], [5, 50, 95]))
fig, axes = plt.subplots(1, 6, figsize=(13, 2.6))
for ax, i in zip(axes, range(6)):
    ax.imshow(Image.open(io.BytesIO(tr_images[i])).convert("RGB"))
    ax.set_title(NAMES[tr_labels[i]], fontsize=8)
    ax.axis("off")
plt.show()

## 2. Transform khớp với weights pretrain

`build_transforms(train, size=160)` trả một `v2.Compose` nhận ảnh PIL, ra tensor `float32 (3, size, size)`:

- **eval**: `Resize(round(size / 0.875))` cạnh ngắn (giữ tỉ lệ, 160 → 183), `CenterCrop(size)`, đổi sang tensor
  `[0, 1]`, `Normalize(IMAGENET_MEAN, IMAGENET_STD)`. Tỉ lệ crop 0.875 là quy ước khi train ImageNet.
- **train**: `RandomResizedCrop(size, scale=(0.35, 1.0))`, `RandomHorizontalFlip()`, rồi đổi sang tensor và
  `Normalize` y như eval. Không lật dọc: ảnh thú nuôi lộn ngược không có trong dữ liệu thật.

Đổi sang tensor: `v2.ToImage()` rồi `v2.ToDtype(torch.float32, scale=True)`.

In [ ]:
def build_transforms(train: bool, size: int = 160):
    # TODO
    raise NotImplementedError

## 3. Model: thay head, đóng băng, learning rate theo nhóm

- `build_model(num_classes, weights=None)`: `torchvision.models.resnet18(weights=weights)`, thay `model.fc`
  bằng `nn.Linear(512, num_classes)` mới.
- `freeze_backbone(model)`: `requires_grad = False` cho mọi tham số trừ `fc`. Đóng băng tham số chưa đủ:
  BatchNorm ở chế độ train vẫn **cập nhật running mean/var** theo dữ liệu mới. Vì vậy thêm
- `set_bn_eval(model)`: đưa mọi `nn.BatchNorm2d` về `eval()` (gọi sau `model.train()` mỗi epoch khi đóng băng).
- `param_groups(model, lr_backbone, lr_head, weight_decay)`: danh sách dict cho optimizer, chỉ gồm tham số có
  `requires_grad`. Tách hai chiều: tham số của `fc` dùng `lr_head`, còn lại `lr_backbone`; tham số 1 chiều
  (bias, `weight`/`bias` của BatchNorm) có `weight_decay` 0, tham số nhiều chiều dùng `weight_decay`.
  Bỏ nhóm rỗng. Mỗi dict có khoá `params`, `lr`, `weight_decay`.

In [ ]:
def build_model(num_classes: int, weights=None) -> nn.Module:
    # TODO
    raise NotImplementedError


def freeze_backbone(model: nn.Module) -> None:
    # TODO
    raise NotImplementedError


def set_bn_eval(model: nn.Module) -> None:
    # TODO
    raise NotImplementedError


def param_groups(model: nn.Module, lr_backbone: float, lr_head: float, weight_decay: float) -> list[dict]:
    # TODO
    raise NotImplementedError

## 4. Feature, vòng train, đánh giá

- `extract_features(model, loader, device="cpu") -> (feats, labels)`: vector 512 chiều **trước** `fc` (sau
  global average pooling) của mọi ảnh, ở chế độ `eval()`, không gradient. `feats` tensor `float32 (N, 512)`,
  `labels` tensor `int64 (N,)`. Không được làm hỏng `model.fc` (khôi phục nếu bạn thay tạm).
- `train_one_epoch(model, loader, optimizer, device="cpu", freeze_bn=False) -> float`: `model.train()`, nếu
  `freeze_bn` thì `set_bn_eval(model)`; cross-entropy; trả loss trung bình **theo mẫu**.
- `evaluate(model, loader, device="cpu") -> float`: top-1 accuracy ở chế độ `eval()`.

In [ ]:
@torch.no_grad()
def extract_features(model: nn.Module, loader: DataLoader, device: str = "cpu") -> tuple[torch.Tensor, torch.Tensor]:
    # TODO
    raise NotImplementedError


def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer, device: str = "cpu", freeze_bn: bool = False) -> float:
    # TODO
    raise NotImplementedError


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: str = "cpu") -> float:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: cấu hình thí nghiệm. GPU: DEVICE = "cuda" (kernel ai-gpu), batch 64 ở 160 px dưới 1.5 GB VRAM
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SIZE, BATCH, WORKERS = 160, 64, 4
torch.manual_seed(0)
train_eval_ds = PetsDataset(tr_images, tr_labels, build_transforms(False, SIZE))   # train nhưng transform eval
train_aug_ds = PetsDataset(tr_images, tr_labels, build_transforms(True, SIZE))
test_ds = PetsDataset(te_images, te_labels, build_transforms(False, SIZE))
loader = lambda ds, shuffle=False: DataLoader(ds, batch_size=BATCH, shuffle=shuffle, num_workers=WORKERS,
                                              generator=torch.Generator().manual_seed(0) if shuffle else None)
x0, _ = test_ds[0]
print(DEVICE, "| một ảnh eval:", tuple(x0.shape), x0.dtype, "mean", round(x0.mean().item(), 3), "std", round(x0.std().item(), 3))

In [ ]:
# explore: (A) linear probe: backbone ImageNet đóng băng, trích feature một lần, train logistic regression trên 512 chiều
W = torchvision.models.ResNet18_Weights.IMAGENET1K_V1          # tải 45 MB vào ~/.cache/torch lần đầu
backbone = build_model(len(NAMES), weights=W)
t0 = time.perf_counter()
f_tr, l_tr = extract_features(backbone, loader(train_eval_ds), DEVICE)
f_te, l_te = extract_features(backbone, loader(test_ds), DEVICE)
t_feat = time.perf_counter() - t0

def fit_linear(f_tr, l_tr, f_te, l_te, epochs=100, lr=1e-2, wd=1e-3, seed=0) -> float:
    """Logistic regression đa lớp (một tầng Linear) trên feature cố định, full-batch AdamW."""
    torch.manual_seed(seed)
    mu, sd = f_tr.mean(0), f_tr.std(0) + 1e-6
    head = nn.Linear(f_tr.shape[1], len(NAMES))
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
    for _ in range(epochs):
        loss = F.cross_entropy(head((f_tr - mu) / sd), l_tr)
        opt.zero_grad()
        loss.backward()
        opt.step()
    with torch.no_grad():
        return float((head((f_te - mu) / sd).argmax(1) == l_te).float().mean())

acc_probe = fit_linear(f_tr, l_tr, f_te, l_te)
print(f"linear probe: test acc {acc_probe:.4f} | trích feature 7 349 ảnh {t_feat:.0f} s")

In [ ]:
# explore: (B) cùng linear probe nhưng quên Normalize ImageNet (chỉ chia 255): train/serve skew kinh điển
no_norm = v2.Compose([v2.Resize(round(SIZE / 0.875)), v2.CenterCrop(SIZE), v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
f_tr_nn, _ = extract_features(backbone, loader(PetsDataset(tr_images, tr_labels, no_norm)), DEVICE)
f_te_nn, _ = extract_features(backbone, loader(PetsDataset(te_images, te_labels, no_norm)), DEVICE)
acc_probe_nonorm = fit_linear(f_tr_nn, l_tr, f_te_nn, l_te)
print(f"linear probe không Normalize: test acc {acc_probe_nonorm:.4f} (đúng: {acc_probe:.4f})")

In [ ]:
# explore: (C) fine-tune toàn bộ: lr backbone nhỏ hơn head 10 lần, augmentation, 3 epoch, cosine
FT_EPOCHS = 3

def run_finetune(weights, lr_backbone, lr_head, epochs=FT_EPOCHS, seed=0) -> tuple[list[float], float, float]:
    torch.manual_seed(seed)
    model = build_model(len(NAMES), weights=weights)
    opt = torch.optim.AdamW(param_groups(model, lr_backbone, lr_head, 0.05))
    dl = loader(train_aug_ds, shuffle=True)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g["lr"] for g in opt.param_groups],
                                                total_steps=epochs * len(dl), pct_start=0.2)
    sched_step = opt.step
    def step_and_schedule():
        sched_step()
        sched.step()
    opt.step = step_and_schedule
    t0 = time.perf_counter()
    losses = [train_one_epoch(model, dl, opt, DEVICE) for _ in range(epochs)]
    return losses, evaluate(model, loader(test_ds), DEVICE), time.perf_counter() - t0

ft_losses, acc_ft, t_ft = run_finetune(W, 1e-4, 1e-3)
print(f"fine-tune: loss theo epoch {[round(l, 3) for l in ft_losses]} | test acc {acc_ft:.4f} | {t_ft:.0f} s")

In [ ]:
# explore: (D) đối chứng: cùng kiến trúc, cùng ngân sách, train từ đầu (weights=None, lr lớn hơn)
sc_losses, acc_scratch, t_sc = run_finetune(None, 1e-3, 1e-3)
print(f"từ đầu: loss theo epoch {[round(l, 3) for l in sc_losses]} | test acc {acc_scratch:.4f} | {t_sc:.0f} s")
print(f"\n{'chiến lược':<28}{'test acc':>9}\n" + "\n".join(f"{k:<28}{v:>9.4f}" for k, v in [
    ("linear probe", acc_probe), ("linear probe, quên Normalize", acc_probe_nonorm),
    (f"fine-tune {FT_EPOCHS} epoch", acc_ft), (f"từ đầu {FT_EPOCHS} epoch", acc_scratch), ("đoán ngẫu nhiên", 1 / len(NAMES))]))

## 5. Đọc kết quả

So số của bạn với bảng ở mục 4 phần lý thuyết. Câu hỏi đáng tự trả lời trước khi làm trắc nghiệm: vì sao chỉ
một tầng Linear trên feature ImageNet đã vượt xa ResNet-18 train từ đầu; vì sao quên `Normalize` làm tụt
accuracy dù backbone và head giống hệt; và khi nào fine-tune toàn bộ đáng thêm thời gian so với linear probe.

Sau khi chạy hết notebook, làm phần **trắc nghiệm** của bài trong app.